In [0]:
%sh
nc -zv c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com 443

Connection to c93366692d52454d9e959ac3f2dc9cb6.eastus.azure.elastic-cloud.com (52.191.218.117) 443 port [tcp/https] succeeded!


In [0]:
%sh curl -s ifconfig.me

52.249.199.78

In [0]:
from sdds.common.util import NotebookUtil
from pyspark.sql.functions import col, lit, coalesce
from pyspark.sql.types import StructType, StructField, StringType
from databricks.sdk.runtime import dbutils

import json
import requests
from requests.auth import HTTPBasicAuth
from datetime import datetime, timedelta

catalog_name = NotebookUtil.notebook_param("sdds_catalog")
schema_name = NotebookUtil.notebook_param("sdds_bronze_schema")
table_name = "catalog-load-inventory-dbx-bronze"

# ES connection config
es_host = "d89a8095f4ec40d8ac0443696bbcb049.eastus.azure.elastic-cloud.com"
es_port = "443"
es_user = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="sdsc-search-es-user-prodauth")
es_password_key = dbutils.secrets.get(scope="kv-dsg-sdsc-dbx-p", key="sdsc-search-es-password-prodauth")
es_index = "catalog-load-read"

## Diagnostics: locate the part-number fields on `inventory` documents

The working `catalog-load-bronze` notebook reads *product* documents where `partnumber` / `parentPartnumber` are top-level fields. This notebook instead filters `type: inventory` documents, which may not carry those top-level fields (the Inventory Backfill notebook keys inventory docs by `id`).

Run the three cells below (in order) to confirm, against the live index, **which field actually holds the part number** before wiring it into the extraction:
1. Inspect the index mapping for any field whose name mentions "part".
2. Dump the full `_source` of a few inventory docs (including `_id`).
3. Probe the candidate part-number fields to see which ones are populated.

In [0]:

body = {
            "bool": {
                "filter": [
                    {"term": {"type": "inventory"}},
                    {
                        "nested": {
                            "path": "inventory",
                            "query": {
                                "bool": {
                                    "filter": [
                                        {
                                            "terms": {
                                                "inventory.location": [0, -1, -2, -4]
                                            }
                                        }
                                    ],
                                    "should": [
                                        {"range": {"inventory.atsqty":  {"gt": 0}}},
                                        {"range": {"inventory.isaqty":  {"gt": 0}}},
                                        {"range": {"inventory.boplqty": {"gt": 0}}},
                                    ],
                                    "minimum_should_match": 1,
                                }
                            },
                        }
                    },
                ]
            }
    }

es_fields = ["partnumber","parentPartnumber","skuInvParentPartNumber",
            "skuInvPartNumber","inventory"]
# Schema: all fields as StringType for bronze-layer raw ingestion
schema = StructType([StructField(f, StringType(), True) for f in es_fields])

# --- Step 1: Scroll ES data and write batches to temp DBFS path ---
base_url = f"https://{es_host}:{es_port}"
auth = HTTPBasicAuth(es_user, es_password_key)
headers = {"Content-Type": "application/json"}

load_ts = datetime.now()
load_timestamp = load_ts.strftime("%Y-%m-%d %H:%M:%S")

# Volume-based landing path partitioned by extraction date (from parameter widget)
volume_base = f"/Volumes/{catalog_name}/{schema_name}/{table_name}"
dbutils.widgets.text("extraction_date", datetime.now().strftime('%Y-%m-%d'))
extraction_date = dbutils.widgets.get("extraction_date")
volume_path = f"{volume_base}/extraction_date={extraction_date}"
dbutils.fs.mkdirs(volume_path)

search_body = {
    "size": 5000,
        "query":body,
    "_source": es_fields
}

response = requests.post(f"{base_url}/{es_index}/_search?scroll=5m", json=search_body, auth=auth, headers=headers)
if not response.ok:
    raise RuntimeError(f"Elasticsearch search failed: {response.status_code} {response.text}")
results = response.json()
scroll_id = results["_scroll_id"]
hits = results["hits"]["hits"]
total_hits = results["hits"]["total"]["value"]
print(f"Total matching documents: {total_hits}")

batch_num = 0
total_fetched = 0

def write_batch(hits_batch, batch_id):
    """Write a batch of hits as newline-delimited JSON to DBFS."""
    records = []
    for hit in hits_batch:
        src = hit["_source"]
        # Convert all values to strings for consistent bronze-layer ingestion
        record = {k: json.dumps(v) if isinstance(v, (list, dict)) else str(v) if v is not None else None for k, v in src.items()}
        records.append(json.dumps(record))
    lines = "\n".join(records)
    dbutils.fs.put(f"{volume_path}/batch_{batch_id:05d}.json", lines, overwrite=True)
    return len(hits_batch)

# Write initial batch
if hits:
    total_fetched += write_batch(hits, batch_num)
    batch_num += 1

# Scroll through remaining results
while len(hits) > 0:
    response = requests.post(f"{base_url}/_search/scroll", json={"scroll": "5m", "scroll_id": scroll_id}, auth=auth, headers=headers)
    response.raise_for_status()
    results = response.json()
    scroll_id = results.get("_scroll_id")
    hits = results["hits"]["hits"]
    if hits:
        total_fetched += write_batch(hits, batch_num)
        batch_num += 1
        if total_fetched % 50000 < 5000:
            print(f"  Fetched {total_fetched} / {total_hits} documents...")

# Clear scroll context
requests.delete(f"{base_url}/_search/scroll", json={"scroll_id": scroll_id}, auth=auth, headers=headers)
print(f"Fetched {total_fetched} documents in {batch_num} batches.")

if total_fetched == 0:
    dbutils.notebook.exit("No new records. Exiting.")





Total matching documents: 685283
Wrote 51616562 bytes.
Wrote 62929220 bytes.
Wrote 64431861 bytes.
Wrote 49332324 bytes.
Wrote 44585472 bytes.
Wrote 47906792 bytes.
Wrote 57210157 bytes.
Wrote 54370929 bytes.
Wrote 52825877 bytes.
Wrote 45231067 bytes.
  Fetched 50000 / 685283 documents...
Wrote 53755493 bytes.
Wrote 53280848 bytes.
Wrote 56122444 bytes.
Wrote 50770214 bytes.
Wrote 47603305 bytes.
Wrote 39625829 bytes.
Wrote 34810438 bytes.
Wrote 35278707 bytes.
Wrote 34922483 bytes.
Wrote 43866193 bytes.
  Fetched 100000 / 685283 documents...
Wrote 58183617 bytes.
Wrote 54172498 bytes.
Wrote 47004402 bytes.
Wrote 49629705 bytes.
Wrote 52694164 bytes.
Wrote 57949479 bytes.
Wrote 55240002 bytes.
Wrote 61560427 bytes.
Wrote 45373149 bytes.
Wrote 48324563 bytes.
  Fetched 150000 / 685283 documents...
Wrote 55337659 bytes.
Wrote 47334052 bytes.
Wrote 68227361 bytes.
Wrote 64165487 bytes.
Wrote 55307757 bytes.
Wrote 70340242 bytes.
Wrote 64982616 bytes.
Wrote 46816354 bytes.
Wrote 50392794 

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7325392259354509>, line 111
    104     dbutils.notebook.exit("No new records. Exiting.")
    106 # --- Step 2: Read JSON files from volume and add load timestamp ---
    107 df = (
    108     spark.read.schema(schema).json(volume_path)
    109     .withColumn("load_timestamp", lit(load_timestamp).cast("timestamp"))
    110     .withColumn("extraction_date", lit(extraction_date).cast("date"))
--> 111     .filter(col("type").isin("style", "sku", "bundle"))
    112 )
    114 record_count = df.count()
    115 print(f"DataFrame record count: {record_count}")

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_even

In [0]:
# --- Step 2: Read JSON files from volume and add load timestamp ---
df = (
    spark.read.schema(schema).json(volume_path)
    .withColumn("load_timestamp", lit(load_timestamp).cast("timestamp"))
    .withColumn("extraction_date", lit(extraction_date).cast("date"))
    .withColumn("partnumber", coalesce(col("partnumber"), col("skuInvPartNumber")))
    .withColumn("parentPartnumber", coalesce(col("parentPartnumber"), col("skuInvParentPartNumber")))
)

record_count = df.count()
print(f"DataFrame record count: {record_count}")
df.display()

DataFrame record count: 685283


partnumber parentPartnumber skuInvParentPartNumber skuInvPartNumber inventory load_timestamp extraction_date 28805330 26YETUHYDR11JFYL47GHS 26YETUHYDR11JFYL47GHS 28805330 [{"location": 1341, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260721042830970"}, {"location": 1583, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260723113156847"}, {"location": 1340, "isaqty": 6, "atsqty": 6, "boplqty": 0, "time": "20260715131547224"}, {"location": 1582, "isaqty": 10, "atsqty": 10, "boplqty": 0, "time": "20260714120411756"}, {"location": 1581, "isaqty": 6, "atsqty": 6, "boplqty": 0, "time": "20260725075649084"}, {"location": 1580, "isaqty": 12, "atsqty": 12, "boplqty": 0, "time": "20260721230138363"}, {"location": 1217, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260726163833043"}, {"location": 1338, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260722022519820"}, {"location": 1579, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260722090643339"}, {"location": 1215, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260626185149837"}, {"location": 1336, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260721140635407"}, {"location": 1214, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260722125052850"}, {"location": 1335, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260629185713468"}, {"location": 1334, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260727130150012"}, {"location": 1333, "isaqty": 11, "atsqty": 11, "boplqty": 0, "time": "20260725105749124"}, {"location": 1574, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260721021518850"}, {"location": 1110, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260715073739912"}, {"location": 1594, "isaqty": 6, "atsqty": 6, "boplqty": 0, "time": "20260725162144384"}, {"location": 1593, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260720041750660"}, {"location": 1592, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260727051522063"}, {"location": 0, "isaqty": 0, "atsqty": 3422, "boplqty": 0, "time": "20260727230115583"}, {"location": 1349, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260714115921396"}, {"location": 1106, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260623184918069"}, {"location": 1348, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260707185643921"}, {"location": 1105, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260626185345623"}, {"location": 1226, "isaqty": 7, "atsqty": 7, "boplqty": 0, "time": "20260704114236239"}, {"location": 1347, "isaqty": 5, "atsqty": 5, "boplqty": 0, "time": "20260706192341279"}, {"location": 1589, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260702185055690"}, {"location": 1588, "isaqty": 5, "atsqty": 5, "boplqty": 0, "time": "20260723141145841"}, {"location": 1223, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260625185511633"}, {"location": 1586, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260724041604351"}, {"location": 1342, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260703190702899"}, {"location": 1584, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260717073844085"}, {"location": 7, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260717022344993"}, {"location": 921, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260622190147663"}, {"location": 4610, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260706192341759"}, {"location": 922, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260626185010608"}, {"location": 924, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260624185958547"}, {"location": 925, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260622185139208"}, {"location": 927, "isaqty": 6, "atsqty": 6, "boplqty": 0, "time": "20260718194255495"}, {"location": 1229, "isaqty": 8, "atsqty": 8, "boplqty": 0, "time": "20260724181514817"}, {"location": 929, "isaqty": 5, "atsqty": 5, "boplqty": 0, "time": "20260726133558711"}, {"location": 1000, "isaqty": 4, "atsqty": 4, "boplqty": 0, "time": "20260727021514218"}, {"lo

In [0]:
# --- Step 3: Write extraction data to Delta table, partitioned by extraction_date (idempotent) ---
target_table = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

# Create table on first run; overwrite only this partition on subsequent runs (idempotent re-runs)
if not spark.catalog.tableExists(f"{catalog_name}.{schema_name}.`{table_name}`"):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("extraction_date")
        .saveAsTable(target_table)
    )
else:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"extraction_date = '{extraction_date}'")
        .partitionBy("extraction_date")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

print(f"Wrote {record_count} records to {target_table} (partition: extraction_date = '{extraction_date}')")

Wrote 685283 records to `dev_sdsc_db`.`sdds_bronze`.`catalog-load-inventory-dbx-bronze` (partition: extraction_date = '2026-07-28')
